In [13]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

C:\Users\marie\AppData\Local\Temp\ipykernel_11336\1757850363.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [54]:
dirpath = os.getcwd()
features_path = r"data\gmfeature_table.csv"
data_path = r"C:\Users\marie\rep_codes\udder_project\udder_analysis\long_format_df"
visit_path = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\milk_videos_visit.csv"
plot_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir),r"adsa\examples")

In [55]:
# only keep inference cows
file_path = r"C:\Users\marie\rep_codes\udder_project\udder_video\filelist_topred.txt"
with open(file_path, "r") as f:
    files = f.read().split("\n")
inf_cows = np.unique([int(file.split(",")[1].split("_")[0]) for file in files])
inf_cows_df = pd.DataFrame(inf_cows, columns = ["cow"])

In [56]:
df = pd.read_csv(os.path.join(data_path, "lactation_features.csv"))
vdf = pd.read_csv(visit_path)
vdf_selected = vdf[['cow', 'days_in_milk']]
df_merged = df.merge(vdf_selected, on = 'cow')
df_merged = inf_cows_df.merge(df_merged, on = "cow", how = "inner")
len(np.unique(df_merged.cow))

93

In [57]:
# add min teat length, max teat length, min eu distance, max eu distance, min gd distance, max gd distance
df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["min_eu"] = [np.nanmin(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_eu"] = [np.nanmax(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]

C:\Users\marie\AppData\Local\Temp\ipykernel_11336\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_11336\1974453648.py:7: RuntimeWarning: All-NaN slice encountered
  df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]


In [58]:
udder_features = ['vol_udder', 'sarea_udder', 'peri_udder', 'area_udder', 'circ_udder', 'exc_udder','min_teat', 'max_teat', 'min_eu', 'max_eu', 'min_gd', 'max_gd']
prod_vars = ['yield_visit_mean', 'interval_sec_mean', 'kickoff_any_perc', 'days_in_milk', "lactation"]

In [59]:
udder_pearson_df = pd.DataFrame(index = udder_features, columns = prod_vars)
udder_pvals_df = pd.DataFrame(index = udder_features, columns = prod_vars)

In [60]:
for u in udder_features:
    for v in prod_vars:
        selected = df_merged[[v, u]].dropna(axis=0) 
        res = stats.pearsonr(selected[u], selected[v])
        udder_pearson_df.loc[u, v] = np.round(res.statistic, 3)
        udder_pvals_df.loc[u, v] = np.round(res.pvalue, 3)

In [61]:
udder_pearson_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.533,0.058,-0.116,0.119,0.406
sarea_udder,0.546,-0.012,-0.175,0.001,0.494
peri_udder,0.602,-0.05,-0.294,0.076,0.648
area_udder,0.58,-0.022,-0.265,0.048,0.636
circ_udder,-0.305,0.017,-0.01,-0.086,-0.297
exc_udder,-0.036,-0.035,0.182,-0.203,0.016
min_teat,0.294,0.162,-0.011,-0.002,0.245
max_teat,0.146,0.049,0.022,-0.016,0.105
min_eu,0.139,0.174,0.102,-0.032,0.094
max_eu,0.474,0.01,-0.161,0.177,0.488


In [62]:
udder_pvals_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.593,0.285,0.272,0.0
sarea_udder,0.0,0.909,0.098,0.991,0.0
peri_udder,0.0,0.632,0.004,0.47,0.0
area_udder,0.0,0.833,0.01,0.649,0.0
circ_udder,0.005,0.883,0.927,0.443,0.007
exc_udder,0.739,0.751,0.093,0.061,0.881
min_teat,0.004,0.119,0.914,0.981,0.018
max_teat,0.161,0.642,0.83,0.881,0.314
min_eu,0.183,0.093,0.328,0.759,0.37
max_eu,0.0,0.925,0.121,0.089,0.0


In [63]:
udder_pvals_df.to_csv(os.path.join(dirpath, "tables", "udder_pvals_df.csv"), index = True)
udder_pearson_df.to_csv(os.path.join(dirpath, "tables", "udder_pearson_df.csv"), index = True)